In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, fbeta_score, accuracy_score
import torch.optim as optim

from transformers.models.bert.modeling_bert import BertOnlyMLMHead


from transformers import (
    BertConfig,
    BertModel,
    AutoConfig,
    PreTrainedModel,
)


from sklearn.metrics import (
    precision_recall_fscore_support,
    fbeta_score,
    accuracy_score,
    classification_report
)

from datetime import datetime
from sklearn.metrics import roc_auc_score


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertForSequenceClassification
import torch
from transformers import AutoModel, AutoTokenizer

# load the tokenizer and saved model safetensors
model_name = "aubmindlab/bert-large-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model architecture and weights
finetuned_model_path = 'load the path to the folder that includes the model.safetensors'
bert_model = BertForSequenceClassification.from_pretrained(finetuned_model_path)


# Move the model to the appropriate device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

# Set the model to evaluation mode
bert_model.eval()

# Start a while loop to check sentences
print("Enter a sentence to check (or type Q to quit):")

while True:
    sentence = input("Sentence: ")
    if sentence.lower() == 'q':
        break

    # Tokenize the input
    inputs = tokenizer(sentence, return_tensors="pt")

    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get the model's predictions
    with torch.no_grad():
        outputs = bert_model(**inputs)
        logits = outputs.logits
        
        prediction = torch.argmax(logits, dim=-1).item()
        if logits.shape[1] != 2:
            raise Exception("Error: the model outputs more than 2 labels, maybe you loaded the wrong model or used wrong architicture")
    # Interpret the result (assuming binary classification: 0 = Not Suicidal, 1 = Suicidal)
    result = "Suicidal" if prediction == 1 else "Not Suicidal"
    print(f"The model predicts: {result}\n")